# Welcome to **UTD**Quake 🌍
[![Docs](https://img.shields.io/badge/docs-UTDQuake-orange)](https://utdquake.readthedocs.io) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97-Dataset-yellow)](https://huggingface.co/datasets/ecastillot/UTDQuake) [![PyPI](https://img.shields.io/pypi/v/utdquake?color=blue)](https://pypi.org/project/utdquake/) [![GitHub](https://img.shields.io/badge/GitHub-UTDQuake-black?logo=github)](https://github.com/ecastillot/UTDQuake)

A global dataset of **earthquakes, stations and seismic phase picks**, ready for machine learning and
seismology.

| | |
|---|---|
| Earthquakes | 346,487 |
| P and S picks | 10.1 million |
| Stations | 17,844 |
| Networks | 82 seismic + 1 DAS |

In this notebook: **1** the dataset · **2** one network · **3** one earthquake · **4** travel times ·
**5** built-in figures.

In [ ]:
!pip install -q utdquake

## 1. The dataset
Each row is a UTDQuake network (`utdq_net`): who reported the events, the catalog, and how much data it has.

In [ ]:
import utdquake as utdq
import matplotlib.pyplot as plt

dataset = utdq.Dataset()
networks = dataset.networks
networks[["utdq_net", "catalog", "country", "events", "p_arrivals", "s_arrivals", "total_stations"]].head(10)

In [ ]:
# where the data come from
by_continent = networks.groupby("continent")[["events", "p_arrivals", "s_arrivals"]].sum()
by_continent.sort_values("events").plot.barh(subplots=True, layout=(1, 3), figsize=(13, 3.5),
                                            legend=False, sharex=False, sharey=True, color="#E87500")
plt.tight_layout()

## 2. One network
`get_network` downloads a network the first time (events, stations, picks and its event bank) and reads
it locally afterwards. Here: **tx**, the Texas seismological network (TexNet).

In [ ]:
net = dataset.get_network("tx")
events, stations, picks = net.events, net.stations, net.picks
print(f"{len(events)} events, {len(stations)} stations, {len(picks)} picks")
events[["time", "latitude", "longitude", "depth", "magnitude", "p_arrivals", "s_arrivals"]].head()

Picks join to stations on `(utdq_net, network, station)`, where `network` is the FDSN code of the station:

In [ ]:
picks = picks.merge(stations, on=["utdq_net", "network", "station"], how="left")
picks[["network", "station", "phase", "time", "distance", "travel_time", "travel_time_zscore"]].head()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# map: events colored by depth, stations as triangles
sc = ax1.scatter(events.longitude, events.latitude, c=events.depth / 1e3, s=12, cmap="viridis_r")
ax1.scatter(stations.confirmed_longitude, stations.confirmed_latitude, marker="^", s=20,
            facecolor="none", edgecolor="k", label="stations")
fig.colorbar(sc, ax=ax1, label="depth (km)")
ax1.set(xlabel="longitude", ylabel="latitude", title="tx: earthquakes and stations")
ax1.legend()

# travel time vs distance, P and S
for phase, color in (("P", "#1f77b4"), ("S", "#E87500")):
    p = picks[picks.phase.str.startswith(phase)]
    ax2.scatter(p.distance * 111.19, p.travel_time, s=1, alpha=0.3, color=color, label=phase)
ax2.set(xlim=(0, 300), ylim=(0, 80), xlabel="epicentral distance (km)", ylabel="travel time (s)",
        title="tx: travel times")
ax2.legend(markerscale=10)
plt.tight_layout()

## 3. One earthquake
Every network comes with an [ObsPy](https://docs.obspy.org) event bank, so you can also get the full
QuakeML events. Here: the largest earthquake and its picks.

In [ ]:
big = events.sort_values("magnitude").iloc[-1]
cat = net.bank.get_events(event_id=big.event_id)
print(cat)

ev = picks[picks.event_id == big.event_id]
fig, ax = plt.subplots(figsize=(7, 5))
for phase, color in (("P", "#1f77b4"), ("S", "#E87500")):
    p = ev[ev.phase.str.startswith(phase)]
    ax.scatter(p.travel_time, p.distance * 111.19, color=color, label=phase)
ax.set(xlabel="time after origin (s)", ylabel="distance (km)",
       title=f"M{big.magnitude:.1f}, {str(big.time)[:19]}")
ax.legend()

## 4. Travel times
Each network has a travel-time model fitted to its picks (median and spread by distance). It is also what
`travel_time_zscore` is measured against, to flag unusual picks.

In [ ]:
import numpy as np

pred = net.travel_time.predict(phase="P", distance=np.arange(1, 200, 5))   # km
fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(pred.distance, pred.travel_time_p1, pred.travel_time_p99, alpha=0.2, label="1-99%")
ax.fill_between(pred.distance, pred.travel_time_p25, pred.travel_time_p75, alpha=0.4, label="25-75%")
ax.plot(pred.distance, pred.travel_time_p50, lw=2, label="median")
ax.set(xlabel="distance (km)", ylabel="P travel time (s)", title="tx: P travel-time model")
ax.legend()

> **Reproducibility.** The dataset is versioned. To always get the data of a given release (e.g. the
> paper's), set `os.environ["UTDQUAKE_REVISION"] = "v1.0"` before downloading.

## 5. Built-in figures
UTDQuake can draw the standard figures of every network. They need [Cartopy](https://cartopy.readthedocs.io)
for the maps.

In [ ]:
!pip install -q cartopy

In [ ]:
net.plot_overview(savepath="tx_overview.png")
net.plot_stats(savepath="tx_stats.png")

**Next:** the [quickstart](https://utdquake.readthedocs.io/en/latest/quickstart.html), the figures of all
networks in the [overview](https://utdquake.readthedocs.io/en/latest/overview.html), or
[add your own catalog](https://utdquake.readthedocs.io/en/latest/upload_dataset.html).

---
**Author:** Emmanuel Castillo, University of Texas at Dallas ·
[LinkedIn](https://www.linkedin.com/in/ecastillot/) · [castillo.280997@gmail.com](mailto:castillo.280997@gmail.com)